# Strict splits and matched-negative candidate design

**Status:** drafting code for review.

**When to run:** after notebook 00 identifies overlap or shortcut risk.

**Inputs:** shared split CSVs. Cold-family, cross-species, temporal, and
matched-negative paths additionally require reviewed metadata files.

**Outputs:** split manifests and candidate review tables only when the
corresponding write switch is enabled.

**Safety:** unknown pairs are treated as *candidates*, never silently as
confirmed negatives. The notebook writes to `splits_ablations/`, never to
the original `splits/` directory.

**Working label definition:** `on-target` means a pair in SNOOPPI's
documented positive split; `off-target` means a pair in its documented
negative split. The unknown split is neither class. Replace these terms
with assay-specific definitions if the project has a narrower target/off-
target ontology, and retain the evidence source for every negative.

## 1. Setup and protocol controls

In [ ]:
from pathlib import Path
import os
import sys

# Keep the repository path separate from SNOOPPI_ROOT, which stores data/artifacts.
# In Colab the recommended clone location is /content/SNOOPPI.
DRAFT_DIR = Path.cwd()
if not (DRAFT_DIR / "snooppi_utils.py").exists():
    repository_root = Path(os.environ.get("SNOOPPI_REPO_ROOT", "/content/SNOOPPI"))
    candidate = repository_root / "drafting_code"
    if candidate.exists():
        DRAFT_DIR = candidate
if not (DRAFT_DIR / "snooppi_utils.py").exists():
    raise FileNotFoundError(
        "Cannot locate drafting_code/snooppi_utils.py. Open the notebook from the "
        "repository clone or set SNOOPPI_REPO_ROOT to the clone root."
    )
if str(DRAFT_DIR) not in sys.path:
    sys.path.insert(0, str(DRAFT_DIR))

from snooppi_utils import (
    LABEL,
    SEQUENCE_A,
    SEQUENCE_B,
    canonical_pair_id,
    ensure_output_directories,
    experiment_paths,
    load_splits,
    normalize_sequence,
    sequence_id,
    split_manifest,
)

PATHS = experiment_paths()
ensure_output_directories(PATHS)
print("SNOOPPI root:", PATHS["root"])

In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit

WRITE_SPLITS = False
WRITE_NEGATIVE_CANDIDATES = False
SEED = 44
TARGET_SIDE = SEQUENCE_B  # Change only if the biological target convention differs.
SPLIT_FRACTIONS = {"train": 0.80, "validation": 0.10, "test": 0.10}
ABLATION_SPLIT_ROOT = PATHS["root"] / "splits_ablations"

source_splits = load_splits(PATHS)
all_pairs = pd.concat(
    [frame.assign(original_split=name) for name, frame in source_splits.items()],
    ignore_index=True,
)
all_pairs["protein_A_id"] = all_pairs[SEQUENCE_A].map(sequence_id)
all_pairs["protein_B_id"] = all_pairs[SEQUENCE_B].map(sequence_id)
all_pairs["pair_id"] = [canonical_pair_id(a, b) for a, b in zip(all_pairs[SEQUENCE_A], all_pairs[SEQUENCE_B])]
conflicting_pair_ids = (
    all_pairs.groupby("pair_id")[LABEL].nunique().loc[lambda values: values.gt(1)].index
)
if len(conflicting_pair_ids):
    raise ValueError(
        f"Found {len(conflicting_pair_ids)} canonical pairs with conflicting labels; "
        "resolve them before creating strict splits."
    )
rows_before_deduplication = len(all_pairs)
all_pairs = all_pairs.drop_duplicates("pair_id", keep="first").reset_index(drop=True)
print("Duplicate canonical pair rows removed:", rows_before_deduplication - len(all_pairs))
print("Pairs available for re-splitting:", len(all_pairs))

## 2. Reusable grouped splitting

`GroupShuffleSplit` keeps one grouping unit in one partition. It does not
guarantee identical prevalence, so every proposed split is summarized and
may need repeated candidate seeds. Selection must occur before model test
results are inspected.

In [ ]:
def grouped_three_way_split(frame, group_column, seed=SEED):
    outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    train_index, heldout_index = next(outer.split(frame, frame[LABEL], groups=frame[group_column]))
    train = frame.iloc[train_index].copy()
    heldout = frame.iloc[heldout_index].copy()

    inner = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
    validation_index, test_index = next(
        inner.split(heldout, heldout[LABEL], groups=heldout[group_column])
    )
    result = {
        "train": train.reset_index(drop=True),
        "validation": heldout.iloc[validation_index].reset_index(drop=True),
        "test": heldout.iloc[test_index].reset_index(drop=True),
    }
    for split_name, part in result.items():
        if set(part[LABEL].unique()) != {0, 1}:
            raise ValueError(
                f"{split_name} contains only one class for group {group_column}; "
                "choose a different prespecified seed or revise the protocol."
            )
    group_sets = {name: set(part[group_column]) for name, part in result.items()}
    assert group_sets["train"].isdisjoint(group_sets["validation"])
    assert group_sets["train"].isdisjoint(group_sets["test"])
    assert group_sets["validation"].isdisjoint(group_sets["test"])
    return result


def summarize_protocol(name, parts, group_column):
    return pd.DataFrame([
        {
            "protocol": name,
            "split": split_name,
            "pairs": len(frame),
            "groups": frame[group_column].nunique(),
            "positive_prevalence": frame[LABEL].mean(),
        }
        for split_name, frame in parts.items()
    ])

## 3. Cold-target split

In [ ]:
target_id_column = "protein_B_id" if TARGET_SIDE == SEQUENCE_B else "protein_A_id"
cold_target = grouped_three_way_split(all_pairs, target_id_column)
cold_target_summary = summarize_protocol("cold_target", cold_target, target_id_column)
display(cold_target_summary)

## 4. Cold-protein connected-component split

If any protein participates in multiple pairs, assigning complete graph
components prevents either partner from crossing split boundaries. A giant
component can make the requested 80/10/10 proportions impossible; the
component-size table makes that limitation visible.

In [ ]:
class UnionFind:
    def __init__(self):
        self.parent = {}

    def find(self, value):
        self.parent.setdefault(value, value)
        if self.parent[value] != value:
            self.parent[value] = self.find(self.parent[value])
        return self.parent[value]

    def union(self, left, right):
        root_left, root_right = self.find(left), self.find(right)
        if root_left != root_right:
            self.parent[root_right] = root_left


components = UnionFind()
for protein_a, protein_b in zip(all_pairs["protein_A_id"], all_pairs["protein_B_id"]):
    components.union(protein_a, protein_b)
all_pairs["interaction_component"] = all_pairs["protein_A_id"].map(components.find)

component_sizes = all_pairs["interaction_component"].value_counts()
display(component_sizes.describe().to_frame("pairs_per_component").round(2))
print("Largest component share:", round(component_sizes.iloc[0] / len(all_pairs), 4))

cold_protein = grouped_three_way_split(all_pairs, "interaction_component")
cold_protein_summary = summarize_protocol("cold_protein", cold_protein, "interaction_component")
display(cold_protein_summary)

## 5. Metadata-dependent split contracts

These protocols require explicit columns. Failing loudly is preferable to
guessing family, species, mutation, or collection date from sequence text.
`protein_metadata.csv` should contain `sequence_sha256` plus available
`family`, `species`, `localization`, `expression_bin`, and `collection_date`.

In [ ]:
METADATA_REQUIREMENTS = {
    "cold_family": ["sequence_sha256", "family"],
    "cross_species": ["sequence_sha256", "species"],
    "temporal": ["sequence_sha256", "collection_date"],
}

if PATHS["metadata"].exists():
    protein_metadata = pd.read_csv(PATHS["metadata"])
    print("Metadata columns:", sorted(protein_metadata.columns))
else:
    protein_metadata = None
    print("Optional metadata not found:", PATHS["metadata"])

def require_metadata(protocol):
    if protein_metadata is None:
        raise FileNotFoundError(f"{protocol} requires {PATHS['metadata']}")
    missing = [column for column in METADATA_REQUIREMENTS[protocol] if column not in protein_metadata]
    if missing:
        raise ValueError(f"{protocol} requires metadata columns: {missing}")

## 6. Matched negative-candidate review

This stage ranks unknown candidate pairs that match positives on available
biology-related strata. It excludes any canonical pair already labeled in
SNOOPPI. The result remains a review queue with `candidate_status=unknown`.
Literature, assay, or trusted database evidence is required before calling
these rows true negatives.

In [ ]:
MATCH_COLUMNS = ["species", "localization", "family", "expression_bin"]

def attach_partner_metadata(frame, metadata):
    result = frame.copy()
    for side, sequence_column in [("A", SEQUENCE_A), ("B", SEQUENCE_B)]:
        result[f"protein_{side}_id"] = result[sequence_column].map(sequence_id)
        renamed = metadata.rename(columns={
            "sequence_sha256": f"protein_{side}_id",
            **{column: f"{side}_{column}" for column in MATCH_COLUMNS if column in metadata},
        })
        keep = [f"protein_{side}_id"] + [f"{side}_{column}" for column in MATCH_COLUMNS if column in metadata]
        result = result.merge(renamed[keep].drop_duplicates(), on=f"protein_{side}_id", how="left", validate="many_to_one")
    return result


candidate_review = None
if PATHS["candidate_negatives"].exists() and protein_metadata is not None:
    candidates = pd.read_csv(PATHS["candidate_negatives"])
    # Candidate files need sequences but intentionally do not need a binary label.
    rename_map = {"Partner_A_sequence": SEQUENCE_A, "Partner_B_sequence": SEQUENCE_B}
    candidates = candidates.rename(columns=rename_map)
    missing = [column for column in (SEQUENCE_A, SEQUENCE_B) if column not in candidates]
    if missing:
        raise ValueError(f"Candidate file is missing: {missing}")
    candidates["pair_id"] = [canonical_pair_id(a, b) for a, b in zip(candidates[SEQUENCE_A], candidates[SEQUENCE_B])]
    candidates = candidates.loc[~candidates["pair_id"].isin(set(all_pairs["pair_id"]))].copy()
    candidates = attach_partner_metadata(candidates, protein_metadata)

    positives = attach_partner_metadata(all_pairs.loc[all_pairs[LABEL].eq(1)].copy(), protein_metadata)
    available_match_columns = [
        f"{side}_{column}"
        for side in ("A", "B")
        for column in MATCH_COLUMNS
        if f"{side}_{column}" in candidates and f"{side}_{column}" in positives
    ]
    if not available_match_columns:
        raise ValueError("No shared matching metadata columns are available")

    # Retain candidates in positive-observed strata; one-to-one selection can be
    # added only after reviewing missingness and candidate counts per stratum.
    positive_strata = positives[available_match_columns].drop_duplicates()
    # Pandas considers missing keys equal during a merge. Exclude incomplete rows
    # so missing biology is never treated as evidence of a match.
    complete_candidates = candidates.dropna(subset=available_match_columns)
    complete_positive_strata = positive_strata.dropna(subset=available_match_columns)
    print("Candidates excluded for missing matching metadata:", len(candidates) - len(complete_candidates))
    candidate_review = complete_candidates.merge(
        complete_positive_strata,
        on=available_match_columns,
        how="inner",
    )
    candidate_review["candidate_status"] = "unknown_requires_review"
    candidate_review["matched_on"] = ",".join(available_match_columns)
    display(candidate_review[["pair_id", "candidate_status", "matched_on"] + available_match_columns].head(20))
    print("Matched unknown candidates available for review:", len(candidate_review))
else:
    print("Candidate matching skipped. Optional inputs:", PATHS["candidate_negatives"], PATHS["metadata"])

## 7. Optional, non-overwriting exports

In [ ]:
def write_protocol(name, parts):
    destination = ABLATION_SPLIT_ROOT / name
    if destination.exists():
        raise FileExistsError(f"Refusing to overwrite {destination}; choose a new protocol name")
    destination.mkdir(parents=True)
    for split_name, frame in parts.items():
        # Remove audit-only hashes before writing model inputs.
        drop_columns = [column for column in ["protein_A_id", "protein_B_id", "pair_id", "interaction_component", "original_split"] if column in frame]
        frame.drop(columns=drop_columns).to_csv(destination / f"{split_name}.csv", index=False)
    print("Saved protocol:", destination)

if WRITE_SPLITS:
    write_protocol("cold_target_seed44_v1", cold_target)
    write_protocol("cold_protein_seed44_v1", cold_protein)
else:
    print("Split previews only. Set WRITE_SPLITS=True after reviewing balance and component sizes.")

if WRITE_NEGATIVE_CANDIDATES:
    if candidate_review is None:
        raise RuntimeError("No candidate review table was built")
    output_path = PATHS["tables"] / "matched_unknown_negative_candidates_v1.csv"
    if output_path.exists():
        raise FileExistsError(f"Refusing to overwrite {output_path}")
    candidate_review.to_csv(output_path, index=False)
    print("Saved review queue, not confirmed negatives:", output_path)

## Next protocols

- **Cold-family:** assign family groups wholly to one partition after the
  family mapping is reviewed; proteins without a family need an explicit
  policy rather than a shared `unknown` group.
- **Mutation:** keep wild-type/mutant relatives in one group during
  training, then evaluate held-out variants and report score deltas.
- **Cross-species:** train on declared source species and evaluate a fully
  held-out species without mixing homologous families when feasible.
- **Temporal:** train only on records dated before a declared cutoff and
  test on later records. Record the source and semantics of the date.